# COVID-19 Chest X-Ray Classification (ResNet-18) — Scientifically Rigorous Version

This notebook trains a **ResNet-18** on the [COVID-19 Radiography Database](https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database) (v5) to classify chest X-rays into 4 classes:
- `COVID`
- `Lung_Opacity`
- `Normal`
- `Viral Pneumonia`

**What changed from the beginner version, and why:**

| Issue in original | Fix here |
|---|---|
| Validation images got random flip/rotation applied | Separate deterministic transform for val/test |
| Only train/val split, no held-out test set | Stratified train/val/**test** split (70/15/15) |
| Hardcoded class weights | Computed automatically from the actual training split |
| Only overall accuracy reported | Per-class precision/recall/F1 + confusion matrix on the **test set**, evaluated once |
| No reproducibility control | Fixed seeds for `torch`, `numpy`, `random` |
| Best model picked by val accuracy | Picked by val **macro-F1** (better for imbalanced classes) |

**Important honesty note:** this notebook gives you a correct, standard *methodology*. It cannot tell you in advance what accuracy you'll get — that depends on your actual data and training run. Don't trust any number (including ones you see quoted in blog posts / Kaggle notebooks claiming 97%+) until you've reproduced it yourself on a proper held-out test set with the exact split you used. Also: **this is a learning exercise, not a validated diagnostic tool** — it should never be treated as a real medical diagnostic system without formal clinical validation, regulatory approval, and radiologist-labeled ground truth review.

## 1. Setup, Imports & Reproducibility

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2. Dataset Indexing

We first scan the folders to build a list of `(path, label)` pairs **without loading images**. This lets us do a proper *stratified* split before any transform is applied, and lets each split (train/val/test) use its own transform.

In [ ]:
CLASSES = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}

# UPDATE THIS PATH!
# Kaggle:  data_dir = '/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset'
# Colab:   data_dir = '/content/COVID-19_Radiography_Dataset'
data_dir = '../data/COVID-19_Radiography_Dataset'

def index_dataset(data_dir):
    """Scan class folders and return parallel lists of image paths and integer labels."""
    paths, labels = [], []
    print("Scanning dataset folders...")
    for class_name in CLASSES:
        class_id = CLASS_TO_IDX[class_name]
        candidates = [
            os.path.join(data_dir, class_name, 'images'),
            os.path.join(data_dir, f"'{class_name}'", 'images'),
            os.path.join(data_dir, class_name),
        ]
        class_dir = next((p for p in candidates if os.path.isdir(p)), None)

        if class_dir is None:
            print(f"WARNING: could not find a folder for class '{class_name}' — check data_dir!")
            continue

        count = 0
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                paths.append(os.path.join(class_dir, fname))
                labels.append(class_id)
                count += 1
        print(f"  {class_name}: {count} images")

    return paths, labels

all_paths, all_labels = index_dataset(data_dir)
print(f"\nTotal images indexed: {len(all_paths)}")

if len(all_paths) == 0:
    raise RuntimeError(
        "No images found. Double-check `data_dir` points at the folder that directly "
        "contains COVID / Lung_Opacity / Normal / Viral Pneumonia subfolders."
    )


## 3. Stratified Train / Validation / Test Split (70 / 15 / 15)

A plain random split (as in the original notebook) can, by chance, skew the class balance between train and val — especially with a minority class as small as `Viral Pneumonia`. We use `stratify=` so every split preserves the original class proportions, and we hold out a **test set that is never touched until final evaluation**.

In [ ]:
# First split off the test set (15%)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    all_paths, all_labels, test_size=0.15, stratify=all_labels, random_state=SEED
)

# Then split the remainder into train (≈70% of total) and val (≈15% of total)
val_fraction_of_remainder = 0.15 / 0.85
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels,
    test_size=val_fraction_of_remainder, stratify=train_val_labels, random_state=SEED
)

print(f"Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}")

# Sanity check: confirm class proportions are preserved across splits
import collections
for name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    counts = collections.Counter(labels)
    total = len(labels)
    dist = {CLASSES[k]: f"{v} ({100*v/total:.1f}%)" for k, v in sorted(counts.items())}
    print(f"{name} distribution: {dist}")


## 4. Dataset Class & Transforms

Separate transforms: **train** gets augmentation, **val/test** are deterministic (resize + normalize only).

In [ ]:
class CovidDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# No augmentation for val/test — we need a clean, deterministic measurement
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset = CovidDataset(train_paths, train_labels, transform=train_transform)
val_dataset = CovidDataset(val_paths, val_labels, transform=eval_transform)
test_dataset = CovidDataset(test_paths, test_labels, transform=eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 5. Model: Pretrained ResNet-18

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, len(CLASSES))
model = model.to(device)
print("Model ready!")


## 6. Class Weights (Computed From Actual Training Data)

We compute these from the **training split's real counts**, not a hardcoded guess — this stays correct even if your local copy of the dataset differs slightly (e.g. duplicates removed, subset used).

In [ ]:
train_counts = collections.Counter(train_labels)
total_train = len(train_labels)
num_classes = len(CLASSES)

class_weights = torch.tensor(
    [total_train / (num_classes * train_counts[i]) for i in range(num_classes)],
    dtype=torch.float
).to(device)

for i, c in enumerate(CLASSES):
    print(f"{c}: count={train_counts[i]}, weight={class_weights[i]:.3f}")


## 7. Training Loop

Model selection is based on **validation macro-F1**, not accuracy — with 4 imbalanced classes, a model can get decent accuracy just by favoring `Normal`. Macro-F1 treats every class equally regardless of its size. We also use early stopping so we don't overfit past the point of real improvement.

In [ ]:
num_epochs = 25
learning_rate = 0.001
patience_for_early_stop = 7

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
best_val_f1 = 0.0
epochs_without_improvement = 0

def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_true = [], []

    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_true.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    accuracy = 100 * correct / total
    macro_f1 = f1_score(all_true, all_preds, average='macro')
    return avg_loss, accuracy, macro_f1


print("Starting training...\n")
for epoch in range(num_epochs):
    train_loss, train_acc, train_f1 = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc, val_f1 = run_epoch(val_loader, train_mode=False)

    scheduler.step(val_f1)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | "
          f"Val Loss {val_loss:.4f} Acc {val_acc:.2f}% Macro-F1 {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "resnet18_covid_best.pth")
        print("  -> New best model saved (by val macro-F1).")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience_for_early_stop:
            print(f"\nEarly stopping: no val macro-F1 improvement for {patience_for_early_stop} epochs.")
            break

print(f"\nBest validation macro-F1: {best_val_f1:.4f}")


## 8. Training Curves

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_ran, history["train_loss"], label="Train Loss")
axes[0].plot(epochs_ran, history["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].set_title("Loss")

axes[1].plot(epochs_ran, history["train_acc"], label="Train Acc")
axes[1].plot(epochs_ran, history["val_acc"], label="Val Acc")
axes[1].plot(epochs_ran, [f*100 for f in history["val_f1"]], label="Val Macro-F1 (x100)", linestyle="--")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("%"); axes[1].legend(); axes[1].set_title("Accuracy / F1")

plt.tight_layout()
plt.show()


## 9. Final Evaluation on the Held-Out Test Set

This is run **exactly once**, using the best checkpoint selected by validation performance. This is what makes the number below a fair, "verified" estimate of generalization — the test set was never seen during training or model selection.

In [ ]:
model.load_state_dict(torch.load("resnet18_covid_best.pth", map_location=device))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_true.extend(labels.tolist())

test_acc = 100 * sum(p == t for p, t in zip(all_preds, all_true)) / len(all_true)
test_macro_f1 = f1_score(all_true, all_preds, average='macro')
print(f"TEST accuracy: {test_acc:.2f}%")
print(f"TEST macro-F1: {test_macro_f1:.4f}\n")

print("Per-class report:")
print(classification_report(all_true, all_preds, target_names=CLASSES, digits=3))


### Confusion Matrix

Shows exactly which classes get confused with each other — critical for this dataset since `COVID` and `Viral Pneumonia` share overlapping radiological patterns (ground-glass opacities).

In [ ]:
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Test Set Confusion Matrix")
plt.tight_layout()
plt.show()


## 10. Summary & Honest Caveats

- The numbers above come from a **single train/val/test split with one random seed**. For a scientifically robust estimate, repeat this with several seeds (or k-fold cross-validation) and report mean ± std, not a single run's number.
- This dataset pools images from multiple hospitals/sources per class (see the dataset's own documentation), which can let a model learn to recognize the *source* (scanner type, image processing) rather than the disease — a known confound in COVID-19 CXR datasets flagged in the literature. Treat any very high accuracy (upper-90s%) with suspicion unless you've checked for this.
- `best_val_f1` and the test metrics above are only meaningful once you've actually run this on your machine — I have not executed this notebook, so I'm not reporting any accuracy number as if it were a real result.
- This model is for learning/prototyping only, not clinical use, without formal validation and regulatory review.